# News IR: full-corpus E5 encoding on a Colab GPU

Open **Runtime → Change runtime type → GPU**, then run cells in order. Try a free GPU first; allocation and session duration are not guaranteed. This notebook performs the same E5-base-v2 encoding tested locally, with checkpoints in your Drive. It does not train a model or create relevance labels.

You need `news_ir_colab.zip` from the project's `artifacts/bundles/` folder and about **6 GB of free Drive space** for embeddings, the final index, and the return archive. The large corpus and weights are downloaded directly inside Colab and checked. Keep the notebook open while working; do not use anti-idle scripts. [Colab resource FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch>=2.9', 'transformers==5.20.1', 'huggingface-hub==2.2.0',
                'faiss-cpu==1.13.2', 'numpy>=2,<3'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU allocated. Select a GPU runtime before proceeding; this cell will not silently use CPU.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## Upload the small code/metadata bundle
Choose `news_ir_colab.zip`, not the corpus or a model file. Re-run this cell after reconnecting to a fresh runtime.


In [ ]:
from google.colab import files
from pathlib import Path
import os, json, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly the news_ir_colab.zip bundle.')
name = next(iter(uploaded))
ROOT = Path('/content/news_ir_task_b')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as archive:
    for member in archive.namelist():
        if ROOT.resolve() not in (ROOT / member).resolve().parents:
            raise ValueError('Unsafe archive member: ' + member)
    archive.extractall(ROOT)
del uploaded
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
os.environ['PYTHONPATH'] = str(ROOT / 'src')
BUNDLE = json.loads((ROOT / 'bundle.json').read_text())
print('Eligible documents:', BUNDLE['eligible_documents'])
print('Pinned E5 revision:', BUNDLE['model_revision'])


## Download and verify the exact corpus and model
The corpus is the same public mirror you used locally. Its SHA-256 must match your original file; a mismatch stops the job. An existing Hugging Face cache may allow interrupted downloads to resume. If the mirror becomes unavailable, upload your original corpus file to this runtime and set `CORPUS` to that file instead.


In [ ]:
from huggingface_hub import hf_hub_download
from news_ir.data import sha256_file
from news_ir.embedding import download_model
CORPUS = Path(hf_hub_download(repo_id=BUNDLE['corpus_repository'], filename='corpus.jsonl',
                            repo_type='dataset', token=False, cache_dir='/content/ir_corpus_cache'))
if sha256_file(CORPUS) != BUNDLE['source_sha256']:
    raise ValueError('Corpus differs from the local BM25 corpus. Use your original file; do not continue with different data.')
MODEL = ROOT / 'model'
spec = download_model(MODEL, revision=BUNDLE['model_revision'])
if spec['files'] != BUNDLE['model_files']:
    raise ValueError('Model files differ from the local tested model.')
# Relocate this COPY of the metadata after checking the complete source hash.
META = ROOT / 'corpus_metadata'
manifest_path = META / 'manifest.json'
metadata = json.loads(manifest_path.read_text())
metadata['source']['path'] = str(CORPUS.resolve())
metadata['source']['mtime_ns'] = CORPUS.stat().st_mtime_ns
manifest_path.write_text(json.dumps(metadata, indent=2) + chr(10))
print('Exact corpus and model verified.')


## Enable persistent checkpoints and measure this GPU
Grant Drive access when prompted. The code uses a dedicated `IR_Project` subfolder. A measured estimate is printed before the full job; it is not a guaranteed runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
JOB = Path('/content/drive/MyDrive/IR_Project') / (BUNDLE['source_sha256'][:12] + '-' + BUNDLE['model_revision'][:12])
JOB.mkdir(parents=True, exist_ok=True)
ENCODINGS = JOB / 'encodings'
DENSE = JOB / 'dense'
RUN = JOB / 'dense-custom-draft'
BATCH_SIZE = 32  # Reduce to 16 or 8 if GPU memory is insufficient.
subprocess.run([sys.executable, 'scripts/benchmark_dense.py', '--document-index', str(META),
                '--model-dir', str(MODEL), '--device', 'cuda', '--batch-size', str(BATCH_SIZE),
                '--count', '128', '--output', str(JOB / 'gpu_benchmark.json')], check=True)


## Encode the complete eligible corpus
This is the long-running cell. Committed shards survive a disconnect in Drive. On a new runtime, repeat the setup/download cells and this cell; it detects the saved config and resumes after the last verified shard. Do not run two sessions against the same checkpoint folder. If memory is insufficient, reduce `BATCH_SIZE` and re-run; finished shards are retained.


In [ ]:
command = [sys.executable, '-m', 'news_ir', 'encode', '--document-index', str(META),
           '--model-dir', str(MODEL), '--corpus', str(CORPUS), '--output', str(ENCODINGS),
           '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--shard-size', '2048']
if (ENCODINGS / 'config.json').exists():
    command.append('--resume')
subprocess.run(command, check=True)
progress = json.loads((ENCODINGS / 'progress.json').read_text())
if progress['coverage'] != 'full' or progress['encoded_documents'] != BUNDLE['eligible_documents']:
    raise RuntimeError('Full corpus is not encoded yet; resume before continuing.')
print(progress)


## Build exact cosine search and export the draft query run
The index uses all eligible documents. The supplied queries remain drafts; these rankings are candidates for review, not relevance labels or final evaluation results.


In [ ]:
if not DENSE.exists():
    subprocess.run([sys.executable, '-m', 'news_ir', 'dense-index', '--encodings', str(ENCODINGS),
                    '--document-index', str(META), '--model-dir', str(MODEL),
                    '--corpus', str(CORPUS), '--output', str(DENSE)], check=True)
queries = ROOT / 'annotations/custom_queries.draft.jsonl'
if sha256_file(queries) != BUNDLE['query_sha256']:
    raise ValueError('Query suite changed unexpectedly.')
if not RUN.exists():
    subprocess.run([sys.executable, '-m', 'news_ir', 'run', '--engine', 'dense', '--index', str(DENSE),
                    '--document-index', str(META), '--model-dir', str(MODEL), '--corpus', str(CORPUS),
                    '--device', 'cuda', '--queries', str(queries), '--output', str(RUN),
                    '--top-k', '1000', '--run-tag', 'e5-custom-draft'], check=True)
print('Full dense index and draft rankings saved in:', JOB)


## Return results to the project
The archive contains the full dense index and saved rankings, not the original corpus or model weights. In Google Drive, open **IR_Project**, then the printed job folder, and download `dense-results.zip` through Drive. Copy it into the project folder and tell the coding assistant it is ready. Keep the checkpoint folder until the local import has been verified.

To extract locally without overwriting existing files: `unzip -n dense-results.zip -d artifacts`. The archive paths are `indexes/dense/...` and `runs/dense-custom-draft/...`. The local CLI already supplies your Mac's BM25 metadata and E5 model as overrides for the paths recorded by Colab.


In [ ]:
archive_path = JOB / 'dense-results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_STORED) as archive:
    for folder, prefix in [(DENSE, 'indexes/dense'), (RUN, 'runs/dense-custom-draft')]:
        for path in folder.rglob('*'):
            if path.is_file():
                archive.write(path, prefix + '/' + str(path.relative_to(folder)))
    archive.write(JOB / 'gpu_benchmark.json', 'dense_gpu_benchmark.json')
print('Download from Google Drive:', archive_path)
print('Archive SHA-256:', sha256_file(archive_path))
print('Archive GB:', round(archive_path.stat().st_size / 1024**3, 2))
